In [1]:
# ============================================================
# STEP 1: LOAD THE SAME DATASET AS METHODS 1-3
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = Path('/content/neo_asteroid_dataset.csv')
if not CSV_PATH.is_file():
    # Uploaded dataset, available in this ChatGPT session.
    local_csv = Path('/mnt/data/neo_asteroid_dataset(1).csv')
    if local_csv.is_file():
        CSV_PATH = local_csv
    else:
        # Google Colab: upload the same CSV used for methods 1-3.
        from google.colab import files
        print('Upload neo_asteroid_dataset.csv')
        uploaded = files.upload()
        csv_files = [Path(p) for p in uploaded if p.lower().endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError('Please upload neo_asteroid_dataset.csv')
        CSV_PATH = next((p for p in csv_files if p.name == 'neo_asteroid_dataset.csv'), csv_files[0])

df = pd.read_csv(CSV_PATH)
print('Dataset shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nPHA counts:')
print(df['pha'].value_counts(dropna=False))
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False))
print('\nFirst five rows:')
print(df.head().to_string(index=False))


Dataset shape: (42815, 21)

Columns:
['spkid', 'pdes', 'full_name', 'kind', 'neo', 'pha', 'e', 'a', 'q', 'i', 'om', 'w', 'moid', 'H', 'diameter', 'albedo', 'first_obs', 'last_obs', 'data_arc', 'n_obs_used', 'condition_code']

PHA counts:
pha
N      40130
Y       2550
NaN      135
Name: count, dtype: int64

Missing values:
albedo            41611
diameter          41569
data_arc            437
pha                 135
moid                135
H                     9
condition_code        2
neo                   0
pdes                  0
spkid                 0
kind                  0
full_name             0
a                     0
w                     0
om                    0
i                     0
q                     0
e                     0
first_obs             0
last_obs              0
n_obs_used            0
dtype: int64

First five rows:
   spkid pdes                full_name kind neo pha      e     a     q     i     om      w   moid     H  diameter  albedo  first_obs   last_o

In [2]:
# ============================================================
# STEP 2: SAME FEATURES, TARGET AND PREPROCESSING AS METHODS 1-3
# ============================================================
FEATURES = ['a', 'e', 'i', 'om', 'w', 'diameter']
TARGET = 'pha'

missing_columns = [col for col in FEATURES + [TARGET] if col not in df.columns]
if missing_columns:
    raise ValueError(f'Missing columns: {missing_columns}')

data = df.dropna(subset=[TARGET]).copy()
data[TARGET] = data[TARGET].astype(str).str.strip().str.upper().map({'Y': 1, 'N': 0})
data = data.dropna(subset=[TARGET]).copy()
data[TARGET] = data[TARGET].astype(int)

for col in FEATURES:
    data[col] = pd.to_numeric(data[col], errors='coerce')

X = data[FEATURES]
y = data[TARGET]
print('Prepared dataset:', data.shape)
print('\nTarget distribution:')
print(y.value_counts())
print('\nMissing values in selected features BEFORE handling:')
print(X.isna().sum())
print('\nDiameter is included in the six model features:', 'diameter' in X.columns)


Prepared dataset: (42680, 21)

Target distribution:
pha
0    40130
1     2550
Name: count, dtype: int64

Missing values in selected features BEFORE handling:
a               0
e               0
i               0
om              0
w               0
diameter    41434
dtype: int64

Diameter is included in the six model features: True


In [3]:
# ============================================================
# STEP 3: SAME STRATIFIED 80/20 SPLIT AS METHODS 1-3
# ============================================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Missing diameter in training:', X_train['diameter'].isna().sum())
print('Missing diameter in testing:', X_test['diameter'].isna().sum())


Training rows: 34144
Testing rows: 8536
Missing diameter in training: 33145
Missing diameter in testing: 8289


In [4]:
# ============================================================
# STEP 4: KNN IMPUTATION + SAME CLASSIFIER AS METHODS 1 AND 3
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import HistGradientBoostingClassifier

# The five orbital variables have different scales. Standardize them
# using TRAINING DATA ONLY so distance-based neighbors are meaningful.
ORBITAL_FEATURES = ['a', 'e', 'i', 'om', 'w']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled.loc[:, ORBITAL_FEATURES] = scaler.fit_transform(X_train[ORBITAL_FEATURES])
X_test_scaled.loc[:, ORBITAL_FEATURES] = scaler.transform(X_test[ORBITAL_FEATURES])

# Fit KNNImputer using only training rows with KNOWN diameters.
# Missing-diameter rows cannot act as donors for diameter. This
# donor-only fit avoids expensive distances between 34k training rows.
# The five orbital features still determine the five nearest neighbors.
donor_mask = X_train_scaled['diameter'].notna()
donors = X_train_scaled.loc[donor_mask, FEATURES]
print('Measured-diameter donors in training:', len(donors))
imputer = KNNImputer(n_neighbors=5, weights='uniform')
imputer.fit(donors)

# Impute test data without ever fitting on the test set.
X_train_prepared = pd.DataFrame(
    imputer.transform(X_train_scaled[FEATURES]), columns=FEATURES, index=X_train.index
)
X_test_prepared = pd.DataFrame(
    imputer.transform(X_test_scaled[FEATURES]), columns=FEATURES, index=X_test.index
)
assert X_train_prepared.notna().sum().sum() == X_train_prepared.size
assert X_test_prepared.notna().sum().sum() == X_test_prepared.size
assert np.allclose(X_train_prepared.loc[donor_mask, 'diameter'], X_train.loc[donor_mask, 'diameter'])

model = HistGradientBoostingClassifier(
    max_iter=100, learning_rate=0.1,
    random_state=42, class_weight='balanced'
)
model.fit(X_train_prepared, y_train)
print('\nMethod 4 model trained successfully!')
print('Missing-value imputation: KNN (5 nearest measured-diameter donors)')
print('Missing train features AFTER imputation:', int(X_train_prepared.isna().sum().sum()))
print('Missing test features AFTER imputation:', int(X_test_prepared.isna().sum().sum()))


Measured-diameter donors in training: 999



Method 4 model trained successfully!
Missing-value imputation: KNN (5 nearest measured-diameter donors)
Missing train features AFTER imputation: 0
Missing test features AFTER imputation: 0


In [5]:
# ============================================================
# STEP 5: SAME EVALUATION METRICS AS METHODS 1-3
# ============================================================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

EVAL_METHOD = 'KNN Imputation'
eval_y = y_test
y_pred = model.predict(X_test_prepared)
y_prob = model.predict_proba(X_test_prepared)[:, 1]

results = {
    'Method': EVAL_METHOD,
    'Training Rows': len(X_train_prepared),
    'Testing Rows': len(X_test_prepared),
    'Test PHA Prevalence': float(eval_y.mean()),
    'Accuracy': accuracy_score(eval_y, y_pred),
    'Precision': precision_score(eval_y, y_pred, zero_division=0),
    'Recall': recall_score(eval_y, y_pred, zero_division=0),
    'F1-score': f1_score(eval_y, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(eval_y, y_prob),
    'Average Precision': average_precision_score(eval_y, y_prob)
}

print('METHOD 4: KNN IMPUTATION')
print('-' * 55)
for metric, value in results.items():
    if isinstance(value, (float, np.floating)):
        print(f'{metric}: {value:.4f}')
    else:
        print(f'{metric}: {value}')
print('\nConfusion Matrix:')
print(confusion_matrix(eval_y, y_pred))
print('\nClassification Report:')
print(classification_report(eval_y, y_pred, target_names=['Non-PHA', 'PHA'], zero_division=0))


METHOD 4: KNN IMPUTATION
-------------------------------------------------------
Method: KNN Imputation
Training Rows: 34144
Testing Rows: 8536
Test PHA Prevalence: 0.0597
Accuracy: 0.7318
Precision: 0.1412
Recall: 0.6863
F1-score: 0.2342
ROC-AUC: 0.7943
Average Precision: 0.1988

Confusion Matrix:
[[5897 2129]
 [ 160  350]]

Classification Report:
              precision    recall  f1-score   support

     Non-PHA       0.97      0.73      0.84      8026
         PHA       0.14      0.69      0.23       510

    accuracy                           0.73      8536
   macro avg       0.56      0.71      0.54      8536
weighted avg       0.92      0.73      0.80      8536



In [6]:
# ============================================================
# STEP 6: SECONDARY FAIR COMPARISON ON THE SAME 247 COMPLETE TEST ROWS
# ============================================================
# Method 6 can evaluate ONLY complete-diameter test rows.
# Show how this method performs on EXACTLY those same test rows too.
common_mask = X_test.notna().all(axis=1)
common_X = X_test_prepared.loc[common_mask]
common_y = y_test.loc[common_mask]
common_pred = model.predict(common_X)
common_prob = model.predict_proba(common_X)[:, 1]
common_results = {
    'Method': EVAL_METHOD,
    'Training Rows': len(X_train_prepared),
    'Testing Rows': len(common_y),
    'Test PHA Prevalence': float(common_y.mean()),
    'Accuracy': accuracy_score(common_y, common_pred),
    'Precision': precision_score(common_y, common_pred, zero_division=0),
    'Recall': recall_score(common_y, common_pred, zero_division=0),
    'F1-score': f1_score(common_y, common_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(common_y, common_prob),
    'Average Precision': average_precision_score(common_y, common_prob)
}
print('\nSECONDARY COMPARISON: SAME COMPLETE-CASE TEST RECORDS')
print('Test rows:', len(common_y))
print('Test PHA prevalence:', round(common_y.mean() * 100, 2), '%')
print('F1-score:', round(common_results['F1-score'], 4))
print('Accuracy:', round(common_results['Accuracy'], 4))
print('NOTE: Training set sizes still differ from complete-case analysis.')



SECONDARY COMPARISON: SAME COMPLETE-CASE TEST RECORDS
Test rows: 247
Test PHA prevalence: 29.96 %
F1-score: 0.5049
Accuracy: 0.587
NOTE: Training set sizes still differ from complete-case analysis.


In [7]:
# ============================================================
# STEP 7: EXPORT RESULTS (SAME METRIC NAMES)
# ============================================================
results_df = pd.DataFrame([results])
results_df.to_csv('04_knn_imputation_results.csv', index=False)
print('Saved: 04_knn_imputation_results.csv')
print(results_df.to_string(index=False))
common_results_df = pd.DataFrame([common_results])
common_results_df.to_csv('04_knn_imputation_complete_test_subset_results.csv', index=False)
print('Saved: 04_knn_imputation_complete_test_subset_results.csv')


Saved: 04_knn_imputation_results.csv
        Method  Training Rows  Testing Rows  Test PHA Prevalence  Accuracy  Precision   Recall  F1-score  ROC-AUC  Average Precision
KNN Imputation          34144          8536             0.059747  0.731842   0.141186 0.686275  0.234192 0.794343           0.198825
Saved: 04_knn_imputation_complete_test_subset_results.csv
